In [ ]:
%cd /kaggle/working
!git clone https://github.com/sontungkieu/RAE
%cd RAE
!git checkout XLA
!curl -LsSf https://astral.sh/uv/install.sh | sh
!ln -s /root/.local/bin/uv /usr/local/bin/uv

In [ ]:
!uv python pin 3.10
!uv init
!uv venv
!uv pip install torch~=2.5.0 torch_xla[tpu]~=2.5.0 torchvision==0.20.1 -f https://storage.googleapis.com/libtpu-releases/index.html --prerelease=allow 1>a.txt 2>b.txt
!uv pip install timm==0.9.16 accelerate==0.23.0 torchdiffeq==0.2.5 wandb scipy torch-fidelity 1>a.txt 2>b.txt
!uv pip install "numpy<2" transformers einops omegaconf pillow wandb 1>a.txt 2>b.txt

In [ ]:
!pip install huggingface_hub

!hf download nyu-visionx/RAE-collections \
  decoders/dinov2/wReg_base/ViTXL_n08/model.pt \
  --local-dir models

!hf download nyu-visionx/RAE-collections \
  stats/dinov2/wReg_base/imagenet1k/stat.pt \
  --local-dir models

!hf download nyu-visionx/RAE-collections \
  discs/dino_vit_small_patch8_224.pth \
  --local-dir models

In [ ]:
import os, pathlib
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
wandb_token  = secrets.get_secret("WANDB2")
hf_tok = secrets.get_secret("HF_TOK_WRITE_KAGGLE")

os.environ["WANDB_API_KEY"] = wandb_token
os.environ["WANDB_KEY"] = wandb_token
os.environ["HF_TOKEN"] = hf_tok
os.environ['PYOPENGL_PLATFORM'] = 'egl'

netrc = pathlib.Path.home() / ".netrc"
netrc.write_text(
    f"machine api.wandb.ai login user password {wandb_token}\n"
)
os.chmod(netrc, 0o600)

In [ ]:
from pathlib import Path
import pandas as pd
import os

src_dir = Path("/kaggle/input/datasets/jessicali9530/celeba-dataset/img_align_celeba/img_align_celeba")
split_csv = Path("/kaggle/input/datasets/jessicali9530/celeba-dataset/list_eval_partition.csv")
out_root = Path("/kaggle/working/celeba256_imgfolder")

df = pd.read_csv(split_csv)
split_map = {0: "train", 1: "val", 2: "test"}  # CelebA standard split

for split in ["train", "val", "test"]:
    (out_root / split / "face").mkdir(parents=True, exist_ok=True)

for _, row in df.iterrows():
    fname = row["image_id"]
    split = split_map[int(row["partition"])]
    src = src_dir / fname
    dst = out_root / split / "face" / fname
    if not dst.exists():
        os.symlink(src, dst)

In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

uv run python src/build_fid_stats.py   --input /kaggle/working/celeba256_imgfolder/val   --output /kaggle/working/celeba256_val_fid_stats.npz   --device cpu   --batch-size 64   --num-workers 32   --num-threads 96


In [ ]:
!cp configs/stage1/training/DINOv2-B_decXL.yaml \
   configs/stage1/training/CelebA256_DINOv2-B_decXL.yaml

!cp configs/stage2/training/ImageNet256/DiTDH-S_DINOv2-B.yaml \
   configs/stage2/training/CelebA256_DiTDH-S_DINOv2-B.yaml

In [ ]:
from pathlib import Path
import textwrap

repo_root = Path("/kaggle/working/RAE")
print("repo_root =", repo_root)

stage1_cfg = repo_root / "configs" / "stage1" / "training" / "CelebA256_DINOv2-B_decXL.yaml"
stage2_cfg = repo_root / "configs" / "stage2" / "training" / "CelebA256_DiTDH-S_DINOv2-B.yaml"

stage1_text = textwrap.dedent("""
stage_1:
  target: stage1.RAE
  params:
    encoder_cls: 'Dinov2withNorm'
    encoder_config_path: 'facebook/dinov2-with-registers-base'
    encoder_input_size: 224
    encoder_params:
      dinov2_path: 'facebook/dinov2-with-registers-base'
      normalize: true
    decoder_config_path: 'configs/decoder/ViTXL'
    pretrained_decoder_path: 'models/decoders/dinov2/wReg_base/ViTXL_n08/model.pt'
    noise_tau: 0.8
    reshape_to_2d: true
    normalization_stat_path: 'models/stats/dinov2/wReg_base/imagenet1k/stat.pt'

training:
  epochs: 16
  ema_decay: 0.9978
  batch_size: 64
  num_workers: 8
  clip_grad: 0.0
  log_interval: 100
  checkpoint_interval: 5000
  sample_interval: 0
  optimizer:
    lr: 2.0e-4
    betas: [0.5, 0.9]
    weight_decay: 0.0
  scheduler:
    type: cosine
    warmup_epochs: 1
    decay_end_epoch: 16
    base_lr: 2.0e-4
    final_lr: 2.0e-5

eval:
  eval_interval: 2500
  eval_model: false
  data_path: '/kaggle/working/celeba256_imgfolder/val'

gan:
  disc:
    arch:
      dino_ckpt_path: 'models/discs/dino_vit_small_patch8_224.pth'
      ks: 9
      norm_type: 'bn'
      using_spec_norm: true
      recipe: 'S_8'
    optimizer:
      lr: 2.0e-4
      betas: [0.5, 0.9]
      weight_decay: 0.0
    scheduler:
      type: cosine
      warmup_epochs: 1
      decay_end_epoch: 16
      base_lr: 2.0e-4
      final_lr: 2.0e-5
    augment:
      prob: 1.0
      cutout: 0.0
    loss:
      disc_loss: hinge
      gen_loss: vanilla
      disc_weight: 0.75
      perceptual_weight: 1.0
      disc_start: 8
      disc_upd_start: 6
      lpips_start: 0
      max_d_weight: 10000.0
      disc_updates: 1
""").strip() + "\n"

stage2_text = textwrap.dedent("""
stage_1:
  target: stage1.RAE
  ckpt: null
  params:
    encoder_cls: 'Dinov2withNorm'
    encoder_config_path: 'facebook/dinov2-with-registers-base'
    encoder_input_size: 224
    encoder_params:
      dinov2_path: 'facebook/dinov2-with-registers-base'
      normalize: true
    decoder_config_path: 'configs/decoder/ViTXL'
    pretrained_decoder_path: 'models/decoders/dinov2/wReg_base/ViTXL_n08/model.pt'
    noise_tau: 0.0
    reshape_to_2d: true
    normalization_stat_path: 'models/stats/dinov2/wReg_base/imagenet1k/stat.pt'

stage_2:
  target: stage2.models.DDT.DiTwDDTHead
  params:
    input_size: 16
    patch_size: 1
    in_channels: 768
    hidden_size: [384, 2048]
    depth: [12, 2]
    num_heads: [6, 16]
    mlp_ratio: 4.0
    class_dropout_prob: 0.1
    num_classes: 1
    use_qknorm: false
    use_swiglu: true
    use_rope: true
    use_rmsnorm: true
    wo_shift: false
    use_pos_embed: true

transport:
  params:
    path_type: 'Linear'
    prediction: 'velocity'
    loss_weight: null
    time_dist_type: 'uniform'

sampler:
  mode: ODE
  params:
    sampling_method: 'euler'
    num_steps: 50
    atol: 1.0e-6
    rtol: 1.0e-3
    reverse: false

guidance:
  method: 'cfg'
  scale: 1.0
  t_min: 0.0
  t_max: 1.0

misc:
  latent_size: [768, 16, 16]
  num_classes: 1
  time_dist_shift_dim: 196608
  time_dist_shift_base: 4096

eval:
  data_path: '/kaggle/working/celeba256_imgfolder/val'
  eval_every: 5000
  batch_size: 4
  num_workers: 0
  max_batches: 32
  eval_model: false
  fid_ref: '/kaggle/working/celeba256_val_fid_stats.npz'
  fid_every: 5000
  fid_num_samples: 4096
  fid_per_proc_batch_size: 4
  fid_batch_size: 128
  fid_device: 'cpu'
  fid_num_threads: 96
  fid_label_sampling: 'random'
  fid_eval_model: false

training:
  global_seed: 0
  epochs: 200
  global_batch_size: 64
  grad_accum_steps: 2
  ema_decay: 0.9995
  num_workers: 0
  log_every: 1
  ckpt_every: 1000000
  sample_every: 1000000
  base_lr: 0.0001
  final_lr: 0.00001
  beta: [0.9, 0.95]
  wd: 0.0
  schedule_type: 'linear'
  decay_start_epoch: 1
  decay_end_epoch: 2
  clip_grad: 1.0
""").strip() + "\n"

stage1_cfg.parent.mkdir(parents=True, exist_ok=True)
stage2_cfg.parent.mkdir(parents=True, exist_ok=True)

stage1_cfg.write_text(stage1_text, encoding="utf-8")
stage2_cfg.write_text(stage2_text, encoding="utf-8")

print("Wrote:", stage1_cfg)
print("Wrote:", stage2_cfg)
# print()
# print("Stage 1 config preview:")
# print(stage1_cfg.read_text()[:1200])
# print()
print("Stage 2 config preview:")
print(stage2_cfg.read_text()[:])

In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

export EXPERIMENT_NAME=celeba256_ditdhxl_stage2_tpu
export ENTITY="TungBangDSLab"
export PROJECT="rae-celeba256-tpu-s-b-dn50-b64-a1-$(TZ=Asia/Bangkok date +%Y%m%d-%H%M%S)"

unset TPU_PROCESS_ADDRESSES || true
unset CLOUD_TPU_TASK_ID || true

echo "WANDB_KEY set? ${WANDB_KEY:+yes}"
echo "HF_TOKEN set? ${HF_TOKEN:+yes}"

uv run python src/train.py \
  --config configs/stage2/training/CelebA256_DiTDH-S_DINOv2-B.yaml \
  --data-path /kaggle/working/celeba256_imgfolder/train \
  --results-dir ckpts/stage2 \
  --image-size 256 \
  --precision bf16 \
  --wandb